# Colab Validation Notebook for ASR Analysis (Ephemeral)

This notebook is configured for a minimal validation run in a temporary Google Colab session.

**IMPORTANT**: All results, including downloaded models and extracted data, will be **deleted** when your Colab session ends. Download any results you want to keep from the file browser before disconnecting.

### How to Use:

1.  **Upload Files to Colab**:
    *   Open [Google Colab](https://colab.research.google.com/).
    *   Go to `File > Upload notebook` and select this file (`Colab_Validation.ipynb`).
    *   In the Colab file browser on the left, click the "Upload" icon and select your `analysis_toolkit.py` file.

2.  **Run the Notebook**:
    *   Run the cells sequentially from top to bottom (`Runtime > Run all`).
    *   The notebook will install libraries and then run the analysis, saving everything to the temporary session storage.

3.  **Download Your Results**:
    *   After the notebook completes, you will find the results in the file browser on the left, inside a folder named `asr_validation_results`.
    *   Right-click the files or folders you want to save and choose "Download".


In [ ]:
# 1. Install All Dependencies
# After running this cell, you MUST restart the runtime.
# Go to Runtime > Restart Runtime in the menu above.
!pip uninstall -y tensorflow
!pip install --upgrade pip -q
!pip install pandas==2.2.2 "numpy<2.0" transformers datasets torch torchaudio scikit-learn tqdm torchcodec librosa -q


---
**➡️ IMPORTANT: Restart the Runtime**
---
Now that the libraries are installed, you must restart the Colab runtime for the changes to take effect.

**Go to the menu and click `Runtime > Restart Runtime`**.

After restarting, you can run the rest of the cells in the notebook.


In [ ]:
# 3. Setup and Configuration
import os
import gc
import pickle
import torch
import pandas as pd
from tqdm import tqdm
import analysis_toolkit as a_t
from IPython.display import display

# === Environment Configuration ===
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# === Path Configuration ===
# Running in an ephemeral instance. Results will be saved to the Colab temporary storage.
# IMPORTANT: Download any results you want to keep before the session ends.
ROOT_DIR = "/content/asr_validation_results"
DATA_DIR = os.path.join(ROOT_DIR, "data")
RESULTS_DIR = os.path.join(ROOT_DIR, "results")
FIG_DIR = os.path.join(ROOT_DIR, "figs")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f"Data will be saved to: {DATA_DIR}")
print(f"Results will be saved to: {RESULTS_DIR}")

# === Model Configuration (Validation Subset) ===
MODELS_TO_ANALYZE = [
    "openai/whisper-base",
    "microsoft/Phi-4-multimodal-instruct",
]

# === Dataset Configuration (Validation Subset) ===
DATASETS_TO_ANALYZE = [
    "PranavBhalerao/l2-arctic-dataset-250",
]

# === Feature Configuration ===
IS_CATEGORICAL_MAP = {
    "gender": True,
    "l1_background": True,
    "duration": False,
    "f0_mean": False,
}


## Section 2: Representation Extraction

This section iterates through the models and datasets defined in the configuration, extracts the hidden layer representations for each, and saves them to disk as pickle files.


In [ ]:
from datasets import load_dataset

for dataset_name in DATASETS_TO_ANALYZE:
    print(f"--- Processing Dataset: {dataset_name} ---")
    dataset_tag = dataset_name.split("/")[-1]
    
    # Load dataset WITHOUT casting audio column - casting causes crashes
    print("Loading dataset (audio will be decoded manually from bytes)...")
    dataset = load_dataset(dataset_name, split="train")
    
    # Use a smaller subset for validation (10 examples for initial testing)
    print(f"Using subset of 10 examples from {len(dataset)} total...")
    dataset_subset = dataset.select(range(min(10, len(dataset))))
    
    # DO NOT cast audio column - the extraction function handles decoding from bytes
    # Casting with Audio() causes std::bad_alloc crashes in Colab

    for model_name in MODELS_TO_ANALYZE:
        print(f"--- Processing Model: {model_name} ---")
        model_tag = model_name.replace("/", "_")
        output_path = os.path.join(DATA_DIR, f"{model_tag}_{dataset_tag}_reps.pkl")
        
        if os.path.exists(output_path):
            print(f"Representations for {model_name} on {dataset_name} already exist. Skipping.")
            continue
        
        try:
            # Load model and processor using the toolkit
            model, processor = a_t.load_model_and_processor(model_name, DEVICE)
            
            # Extract representations using the toolkit
            reps_by_layer, labels = a_t.extract_representations(model, processor, dataset_subset, DEVICE)
            
            # Save the extracted data
            print(f"Saving representations to {output_path}...")
            with open(output_path, "wb") as f:
                pickle.dump({
                    "reps_by_layer": reps_by_layer,
                    "labels": labels
                }, f)
            print("Save complete.")
            
        except Exception as e:
            print(f"Failed to process {model_name} on {dataset_name}. Error: {e}")
        
        # Clean up memory
        if 'model' in locals(): del model
        if 'processor' in locals(): del processor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

print("\n--- Representation extraction complete for all models and datasets. ---")

## Section 3: Core Probing Analysis

This section loads the saved representation files and runs the main linear probing analysis for each model-dataset pair. The results are aggregated into a single DataFrame.


In [ ]:
all_results = []

for dataset_name in DATASETS_TO_ANALYZE:
    dataset_tag = dataset_name.split("/")[-1]
    
    for model_name in MODELS_TO_ANALYZE:
        model_tag = model_name.replace("/", "_")
        data_path = os.path.join(DATA_DIR, f"{model_tag}_{dataset_tag}_reps.pkl")
        results_path = os.path.join(RESULTS_DIR, f"{model_tag}_{dataset_tag}_probing_results.csv")

        if not os.path.exists(data_path):
            print(f"Representation file for {model_name} on {dataset_name} not found. Skipping.")
            continue

        if os.path.exists(results_path):
            print(f"Results for {model_name} on {dataset_name} already exist. Loading from disk.")
            model_results_df = pd.read_csv(results_path)
            all_results.append(model_results_df)
            continue

        try:
            print(f"Loading representations for {model_name} on {dataset_name} from {data_path}...")
            with open(data_path, "rb") as f:
                data = pickle.load(f)
            
            reps_by_layer = data["reps_by_layer"]
            labels = data["labels"]

            # Run probing analysis using the toolkit
            print(f"Running probing analysis for {model_name} on {dataset_name}...")
            model_results_df = a_t.run_probing_analysis(reps_by_layer, labels, IS_CATEGORICAL_MAP)
            model_results_df["model_name"] = model_name
            model_results_df["dataset"] = dataset_name
            
            # Save results
            print(f"Saving probing results to {results_path}...")
            model_results_df.to_csv(results_path, index=False)
            all_results.append(model_results_df)

        except Exception as e:
            print(f"Failed to run probing for {model_name} on {dataset_name}. Error: {e}")

# Aggregate all results
if all_results:
    final_results_df = pd.concat(all_results, ignore_index=True)
    final_output_path = os.path.join(RESULTS_DIR, "all_models_all_datasets_probing_results.csv")
    final_results_df.to_csv(final_output_path, index=False)
    print(f"\n--- All probing results aggregated and saved to {final_output_path} ---")
    display(final_results_df.head())
else:
    print("\nNo results were generated.")
